In [1]:
import ee
import geemap
import geopandas as gpd
import pprint as pp

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')

date = '2020-05-29'
date_plus1d = '2020-05-30'
roi_name = 'YKF_sub3'
image_footprints_path = f'./data/overlap_dates_for_roi/{roi_name}_overlap_dates.shp'
best_image_dates = gpd.read_file(image_footprints_path) 
est_utm = f'EPSG:{best_image_dates.estimate_utm_crs().to_epsg()}'

idx = 0

def convert_gpd_geom_to_ee(geom, est_utm):
    """
    Takes a geopandas geom object and coverts it to an Earth Engine polygon
    """
    if est_utm is None:
        out_crs = 'EPSG:4326'
    else:
        out_crs = est_utm

    coords = list(geom.exterior.coords)
    coords_list = [[x, y] for x, y in coords]
    return ee.Geometry.Polygon(coords_list, proj=out_crs)

polygon = convert_gpd_geom_to_ee(best_image_dates.geometry[idx], None)

In [2]:
def rescale_s2(img):
    rescaled_bands = img.divide(10_000)
    return rescaled_bands
    
def rescale_ls8(img):
    rescaled_bands = img.multiply(0.0000275).add(-0.2)
    return rescaled_bands

In [7]:
# 1. Reference Landsat image and its projection
ls_col = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
    .filterDate(date, date_plus1d) \
    .filterBounds(polygon) \
    .select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5']) \
    .first()

ls = rescale_ls8(ls_col)


# 2. Sentinel-2 ImageCollection mosaic
s2_col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterDate(date, date_plus1d) \
    .filterBounds(polygon) \
    .select(['B2', 'B3', 'B4', 'B8']) \
    .first() 


s2 = rescale_s2(s2_col)


In [11]:
ls_og_proj = ls.projection().getInfo()

ls_utm = ls.reproject(crs=ls_og_proj['crs'], crsTransform=ls_og_proj['transform']).resample('bilinear')
ls_utm_proj = ls_utm.projection().getInfo()

s2_utm = s2.reproject(crs=ls_utm_proj['crs'], crsTransform=ls_utm_proj['transform']).resample('bilinear')

pp.pp(ls.projection().getInfo())
pp.pp(s2.projection().getInfo())
pp.pp(ls_utm.projection().getInfo())
pp.pp(s2_utm.projection().getInfo())

{'type': 'Projection',
 'crs': 'EPSG:32607',
 'transform': [30, 0, 254385, 0, -30, 7556115]}
{'type': 'Projection',
 'crs': 'EPSG:32606',
 'transform': [10, 0, 600000, 0, -10, 7400040]}
{'type': 'Projection',
 'crs': 'EPSG:32607',
 'transform': [30, 0, 254385, 0, -30, 7556115]}
{'type': 'Projection',
 'crs': 'EPSG:32607',
 'transform': [30, 0, 254385, 0, -30, 7556115]}


In [5]:
ls_utm_proj = ls_utm.projection()

write_ls = ee.batch.Export.image.toDrive(
    image=ls_utm,
    description=f'ls_image{idx}',
    fileNamePrefix=f'ls_image{idx}',
    folder='test2',
    crs=ls_utm_proj.getInfo()['crs'],
    crsTransform=ls_utm_proj.getInfo()['transform'],
    region=polygon,
    maxPixels=1e13
)

write_s2 = ee.batch.Export.image.toDrive(
    image=s2_utm,
    description=f's2_image{idx}',
    fileNamePrefix=f's2_image{idx}',
    folder='test2',
    crs=ls_utm_proj.getInfo()['crs'],
    crsTransform=ls_utm_proj.getInfo()['transform'],
    region=polygon,
    maxPixels=1e13
)

# write_ls.start()
# write_s2.start()


In [12]:
Map = geemap.Map(center=[0, 0], zoom=2)

# Visualization parameters for Landsat 8
# Adjust min/max to match your rescaling
vis_params_ls = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],  # Red, Green, Blue
    'min': 0.0,
    'max': 0.3
}

# Visualization parameters for Sentinel-2
vis_params_s2 = {
    'bands': ['B4', 'B3', 'B2'],          # Red, Green, Blue
    'min': 0.0,
    'max': 0.3
}

# Add the layers to the map
Map.addLayer(ls, vis_params_ls, "Landsat 8")
Map.addLayer(s2, vis_params_s2, "Sentinel-2")
Map.addLayer(ls_utm, vis_params_ls, "Landsat 8 (Reprojected UTM)")
Map.addLayer(s2_utm, vis_params_s2, "Sentinel-2 (Reprojected UTM)")


# Display the map
Map


Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…